# **Initialization**

In [1]:
print('Start')

Start


In [3]:
#%load_ext autoreload
%reload_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import math
import random
import sys
import pulp
import vrplib
import re
import sys
import os
import gc
import contextlib
import modified_didppy as m_dp
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.optimize import linear_sum_assignment
from numpy.linalg import eigh
import time
from ortools.linear_solver import pywraplp
from functools import lru_cache

# --- 1. DEFINE PATH TO LIBRARY PARENT FOLDER ---
# Replace this with the ACTUAL path to the folder containing 'didp_ea_lib'
# IMPORTANT: Use r"..." string to handle Windows backslashes correctly
LIBRARY_PARENT_PATH = r"C:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\THESIS_MODIFIED_DIDP\Evolutionary_algorithm"
# --- 2. ADD TO SYSTEM PATH ---
if LIBRARY_PARENT_PATH not in sys.path:
    sys.path.append(LIBRARY_PARENT_PATH)
print(f"Library path added: {LIBRARY_PARENT_PATH}")
# --- 3. TEST IMPORT ---
try:
    import evolutionary_algorithm_lib
    from evolutionary_algorithm_lib import *
    from evolutionary_algorithm_lib import (compile_chromosome_to_useable_function, 
                                            combining_modified_didppy_solver_with_chromosome)
    from evolutionary_algorithm_lib.utils import automatic_creation_of_dual_bounds_registry
    
    print("✅ Success! 'evolutionary_algorithm_lib' is imported and ready.")
except ImportError as e:
    print(f"❌ Error: Could not import library. Check the path above.\nDetails: {e}")

Library path added: C:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\THESIS_MODIFIED_DIDP\Evolutionary_algorithm
✅ Success! 'evolutionary_algorithm_lib' is imported and ready.


# **Data**

In [4]:
# 1. Define your directory path
# TIP: Use r"..." (raw string) so Python treats backslashes as text, not escape characters
base_path = r"C:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\3_CVRP_dual_bounds_and_models\Datasets\A"

# 2. Construct the full file paths
# We assume the solution file has the standard .sol extension
instance_path = os.path.join(base_path, "A-n32-k5.vrp")

# 3. Load the data
try:
    # Read the instance data
    instance = vrplib.read_instance(instance_path)
    solution = vrplib.read_solution(instance_path.replace('.vrp', '.sol'))
    # 4. Print results to verify
    print(f"Successfully loaded: {instance['name']}")
    print(f"Number of customers: {instance['dimension']}")
    print(f"Vehicle Capacity: {instance['capacity']}")
    print(f"Optimal solution cost (from .sol file): {solution['cost']}")
    # 'instance' is the dictionary you provided in the prompt

except Exception as e:
    print(e)



Successfully loaded: A-n32-k5
Number of customers: 32
Vehicle Capacity: 100
Optimal solution cost (from .sol file): 784


In [5]:
# 1. Extract necessary parameters
current_capacity = instance['capacity'] # 100
current_num_locations = instance['dimension'] # 53
match = re.search(r"No of trucks:\s*(\d+)", instance['comment'])
current_optimal_cost = solution['cost']  # Extract optimal cost from solution file  
if match:
    current_num_vehicles = int(match.group(1))
else:
    print("Number of trucks not found.")
    
# 2. Extract Demands
# Note: instance['demand'] includes the Depot at index 0 (value 0)
# This is usually what you want for 0-based indexing in state representation
current_cust_demands = instance['demand'] 

# 3. Extract and FIX the Distance Matrix
# The library calculated exact Euclidean distances (floats). 
# For the 'A' (Augerat) series, we typically round to the nearest integer.
current_travel_cost = instance['edge_weight']

# --- VERIFICATION ---
print(f"Capacity: {current_capacity}")
print(f"Number of Nodes: {current_num_locations}")
print(f"Number of Vehicles: {current_num_vehicles}")
print(f"Depot Demand: {current_cust_demands[0]}")
print(f"Customer 1 Demand: {current_cust_demands[1]}")
print(f"Best solution cost: {current_optimal_cost}")
print("\nComparison of Distance (Depot -> Node 1):")
print(f"Distance matrix: {current_travel_cost}") 

Capacity: 100
Number of Nodes: 32
Number of Vehicles: 5
Depot Demand: 0
Customer 1 Demand: 19
Best solution cost: 784

Comparison of Distance (Depot -> Node 1):
Distance matrix: [[  0.          34.92849839  77.87810989 ...  62.28964601  16.2788206
   72.78049189]
 [ 34.92849839   0.          60.30754513 ...  80.32434251  19.41648784
   39.05124838]
 [ 77.87810989  60.30754513   0.         ...  71.58910532  65.19202405
   48.        ]
 ...
 [ 62.28964601  80.32434251  71.58910532 ...   0.          65.76473219
  101.53324579]
 [ 16.2788206   19.41648784  65.19202405 ...  65.76473219   0.
   56.5154846 ]
 [ 72.78049189  39.05124838  48.         ... 101.53324579  56.5154846
    0.        ]]


# **1. Model and dual bound declaration**

In [6]:
def creation_of_didp_model_function():
    """
    Creates the CVRP DIDP model and returns it along with necessary metadata 
    for the heuristic functions.
    """
    # =========================================================
    # 1. Define Data
    # =========================================================
    n = current_num_locations
    m = current_num_vehicles
    q = current_capacity
    # Weights (demand)
    d = current_cust_demands

    # Distance matrix
    distance_list = current_travel_cost
    
    # =========================================================
    # 2. Define DIDP model
    # =========================================================
    model = m_dp.Model(float_cost= True)

    customer = model.add_object_type(number=n)
    unvisited_var = model.add_set_var(object_type=customer, target=list(range(1, n)), name='unvisited_customers')
    location_var = model.add_element_var(object_type=customer, target=0)
    load_var = model.add_float_resource_var(target=0, less_is_better=True)
    vehicles_var = model.add_int_resource_var(target=1, less_is_better=True)

    weight = model.add_float_table(d)
    distance_table = model.add_float_table(distance_list)

    model.add_base_case([unvisited_var.is_empty(), location_var == 0])

    for j in range(1, n):
        visit = m_dp.Transition(
            name=f"visit {j}",
            cost=distance_table[location_var, j] + m_dp.FloatExpr.state_cost(),
            effects=[
                (unvisited_var, unvisited_var.remove(j)),
                (location_var, j),
                (load_var, load_var + weight[j]),
            ],
            preconditions=[unvisited_var.contains(j), load_var + weight[j] <= q],
        )
        model.add_transition(visit)

    for j in range(1, n):
        visit_via_depot = m_dp.Transition(
            name=f"visit {j} with new vehicle",
            cost=distance_table[location_var, 0] + distance_table[0, j] + m_dp.FloatExpr.state_cost(),
            effects=[
                (unvisited_var, unvisited_var.remove(j)),
                (location_var, j),
                (load_var, weight[j]),
                (vehicles_var, vehicles_var + 1),
            ],
            preconditions=[unvisited_var.contains(j), vehicles_var < m],
        )
        model.add_transition(visit_via_depot)

    return_to_depot = m_dp.Transition(
        name="return",
        cost=distance_table[location_var, 0] + m_dp.FloatExpr.state_cost(),
        effects=[(location_var, 0)],
        preconditions=[unvisited_var.is_empty(), location_var != 0],
    )
    model.add_transition(return_to_depot)

    model.add_state_constr((m - vehicles_var + 1) * q - load_var >= weight[unvisited_var])

    # =========================================================
    # 3. Bundle Metadata
    # =========================================================
    metadata = {
        "unvisited_var": unvisited_var,
        "location_var": location_var,
        "distance_matrix": distance_list,
        "demand": d,
        "capacity": q,
        "num_vehicles": m,
        "num_nodes": n
    }
    
    didp_bundle = (model, metadata)
    return didp_bundle

In [7]:
def create_persistent_lp_relaxation_3_index_dual_bounds(metadata):
    """
    Creates a persistent 3-Index (Vehicle-Node-Node) CVRP Relaxation.
    - ADHERES TO: Model VRP4 (Equations 1.28 - 1.38 in PDF).
    - SOLVER: Google OR-Tools (GLOP).
    - PERSISTENT: Creates model once, toggles 'y' variable bounds to activate/deactivate nodes.
    """
    # --- Extract Static Data ---
    n_nodes = metadata['num_nodes']
    n_vehicles = metadata['num_vehicles']
    capacity = metadata['capacity']
    demands = metadata['demand']
    dist_matrix = metadata['distance_matrix']
    
    unvisited_var = metadata['unvisited_var']
    location_var = metadata['location_var']

    # ==========================================
    # 1. INITIALIZATION (Runs Once)
    # ==========================================
    solver = pywraplp.Solver.CreateSolver('GLOP')
    if not solver:
        return lambda state: 0.0

    infinity = solver.infinity()

    # --- Variables ---
    # x[k, i, j]: Vehicle k traverses arc (i, j)
    # y[i, k]: Customer i is served by vehicle k
    # u[i, k]: Load of vehicle k after visiting customer i
    x = {}
    y = {}
    u = {}

    # 1. Create x_ijk (Flow)
    for k in range(n_vehicles):
        for i in range(n_nodes):
            for j in range(n_nodes):
                if i != j:
                    x[(k, i, j)] = solver.NumVar(0, 1, f'x_{k}_{i}_{j}')

    # 2. Create y_ik (Assignment)
    # Defined for ALL nodes (0..N) and ALL vehicles (1..K)
    for i in range(n_nodes):
        for k in range(n_vehicles):
            y[(i, k)] = solver.NumVar(0, 1, f'y_{i}_{k}')

    # 3. Create u_ik (Potentials for MTZ)
    # Defined for Customers (1..N) and Vehicles (1..K)
    # The requirement d_i <= u_ik <= C will be relaxeddynamically
    for i in range(1, n_nodes):
        for k in range(n_vehicles):
            u[(i, k)] = solver.NumVar(0, capacity, f'u_{i}_{k}')

    # --- Constraints ---
    
    # Store references to "Assignment" constraints to toggle them later
    cons_assignment = {} 

    # (1.29) Customer Assignment: Each customer i is visited exactly once
    # sum_{k} y_{ik} = 1
    # We will toggle the RHS of this constraint between 1 (Active) and 0 (Inactive)
    for i in range(1, n_nodes):
        c = solver.Constraint(0, 0, f'assign_{i}') # Default 0 (Inactive)
        for k in range(n_vehicles):
            c.SetCoefficient(y[(i, k)], 1)
        cons_assignment[i] = c

    # (1.30) Depot Usage: K vehicles leave the depot
    # sum_{k} y_{0k} = K (Relaxed to <= K for lower bound purposes)
    c_depot = solver.Constraint(0, n_vehicles, 'depot_usage')
    for k in range(n_vehicles):
        c_depot.SetCoefficient(y[(0, k)], 1)

    # (1.31) Flow Conservation & Link to Assignment
    # sum_{j} x_{ijk} = y_{ik}  AND  sum_{j} x_{jik} = y_{ik}
    # This ensures if y_ik=0, no flow for k touches i. If y_ik=1, flow must enter and leave.
    for k in range(n_vehicles):
        for i in range(n_nodes):
            # Outgoing Flow: sum_{j} x_{ijk} - y_{ik} = 0
            c_out = solver.Constraint(0, 0, f'flow_out_{i}_{k}')
            c_out.SetCoefficient(y[(i, k)], -1)
            for j in range(n_nodes):
                if i != j:
                    c_out.SetCoefficient(x[(k, i, j)], 1)
            
            # Incoming Flow: sum_{j} x_{jik} - y_{ik} = 0
            c_in = solver.Constraint(0, 0, f'flow_in_{i}_{k}')
            c_in.SetCoefficient(y[(i, k)], -1)
            for j in range(n_nodes):
                if i != j:
                    c_in.SetCoefficient(x[(k, j, i)], 1)

    # (1.37) & (1.38) MTZ Subtour Elimination & Capacity
    # u_{ik} - u_{jk} + C * x_{ijk} <= C - d_j
    # Valid for all i, j in {1..N} (Customers), i != j, all k
    for k in range(n_vehicles):
        for i in range(1, n_nodes):
            for j in range(1, n_nodes):
                if i != j:
                    # RHS: C - d_j
                    # LHS: u_{ik} - u_{jk} + C * x_{ijk}
                    c = solver.Constraint(-infinity, float(capacity - demands[j]), f'mtz_{k}_{i}_{j}')
                    c.SetCoefficient(u[(i, k)], 1)
                    c.SetCoefficient(u[(j, k)], -1)
                    c.SetCoefficient(x[(k, i, j)], capacity)
    
    # Bound Coordination (1.38): d_i <= u_{ik} <= C
    # This is handled by variable bounds, but we link u_{ik} to y_{ik} logic implicitly
    # If y_{ik}=0, then x=0, and the constraint becomes u_i - u_j <= C - d_j (loose).
    # To be strictly precise with VRP4, we should force u_{ik}=0 if y_{ik}=0, 
    # but for a relaxation loose bounds are acceptable and faster.

    # --- Objective ---
    # Minimize sum_{i,j,k} c_{ij} * x_{ijk}
    objective = solver.Objective()
    for k in range(n_vehicles):
        for i in range(n_nodes):
            for j in range(n_nodes):
                if i != j:
                    objective.SetCoefficient(x[(k, i, j)], dist_matrix[i][j])
    objective.SetMinimization()

    # ==========================================
    # 2. HEURISTIC FUNCTION (Runs per State)
    # ==========================================
    @lru_cache(maxsize=10000)
    def h_lp_relaxation_3_idx(state):
        unvisited = state[unvisited_var]
        current_loc = state[location_var]
        
        # Quick exit
        if not unvisited and current_loc == 0: 
            return 0.0

        # Define Active Set: Unvisited + Current + Depot
        active_customers = set(unvisited)
        if current_loc != 0:
            active_customers.add(current_loc)
        
        # --- Update Model (Toggle Constraints) ---
        
        # 1. Update Customer Assignment Constraints (1.29)
        # We iterate through ALL customers 1..N
        for i in range(1, n_nodes):
            if i in active_customers:
                # ACTIVE: Must be visited exactly once
                cons_assignment[i].SetBounds(1, 1)
                
                # Activate u variables (Logic: d_i <= u <= C)
                # Note: In 3-index, u is per vehicle. We just set bounds.
                for k in range(n_vehicles):
                    u[(i, k)].SetBounds(demands[i], capacity)
                    
            else:
                # INACTIVE: Must NOT be visited
                cons_assignment[i].SetBounds(0, 0)
                
                # Deactivate u variables
                for k in range(n_vehicles):
                    u[(i, k)].SetBounds(0, 0)

        # 2. Anchoring (Heuristic improvement)
        # If we are at 'current_loc', we effectively treat it as a "start" for ONE vehicle.
        # However, in VRP4, all vehicles start at 0.
        # To strictly follow VRP4 on the subgraph, we treat the current problem as 
        # "Routing K vehicles to cover the set {Unvisited U Current}".
        # This gives a valid Lower Bound (Relaxation).
        
        # Optional: Dynamic Bound Tightening (4.2)
        # For the 3-index model, 'u' represents LOAD.
        # We can't strictly anchor u=0 because in VRP4 u represents accumulated load, 
        # and we don't know the load of the vehicle arriving at current_loc in the LP.
        # So we leave standard bounds [d_i, C].

        # Solve
        status = solver.Solve()
        
        if status == pywraplp.Solver.OPTIMAL:
            return float(objective.Value())
        return 0.0

    return h_lp_relaxation_3_idx

In [8]:
def create_persistent_lp_relaxation_2_index_dual_bounds(metadata):
    """
    Creates a persistent 2-Index (Flow-based) CVRP Relaxation.
    - SCALE: Capable of handling N=100 in < 0.1s per node.
    - LOGIC: Drops 'Vehicle' dimension. Treats flow as aggregate.
    - SOLVER: Google OR-Tools (GLOP).
    """
    # --- Extract Static Data ---
    n_nodes = metadata['num_nodes']
    n_vehicles = metadata['num_vehicles']
    capacity = metadata['capacity']
    demands = metadata['demand']
    dist_matrix = metadata['distance_matrix']
    unvisited_var = metadata['unvisited_var']

    # ==========================================
    # 1. INITIALIZATION (Runs Once)
    # ==========================================
    solver = pywraplp.Solver.CreateSolver('GLOP')
    if not solver:
        return lambda state: 0.0

    infinity = solver.infinity()

    # --- Variables ---
    # x[i, j]: Binary flow from i to j (Aggregated over all vehicles)
    x = {}
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j:
                x[(i, j)] = solver.NumVar(0, 1, f'x_{i}_{j}')

    # u[i]: Accumulated load variable for MTZ
    u = {i: solver.NumVar(0, capacity, f'u_{i}') for i in range(1, n_nodes)}

    # --- Constraints ---
    cons_degree_out = {} # Outgoing degree 
    cons_degree_in = {}  # Incoming degree

    # 1. Degree Constraints (Customers 1..N)
    # Each active customer must have exactly 1 outgoing and 1 incoming edge
    for i in range(1, n_nodes):
        # Outgoing
        c_out = solver.Constraint(0, 0, f'deg_out_{i}')
        for j in range(n_nodes):
            if i != j:
                c_out.SetCoefficient(x[(i, j)], 1)
        cons_degree_out[i] = c_out

        # Incoming
        c_in = solver.Constraint(0, 0, f'deg_in_{i}')
        for j in range(n_nodes):
            if i != j:
                c_in.SetCoefficient(x[(j, i)], 1)
        cons_degree_in[i] = c_in

    # 2. Depot Degree Constraints (Static)
    # Total outgoing flow from Depot == Number of active vehicles (<= K)
    # We relax this to <= K for lower bound purposes
    c_depot_out = solver.Constraint(0, n_vehicles, 'depot_out')
    for j in range(1, n_nodes):
        c_depot_out.SetCoefficient(x[(0, j)], 1)
    
    # Total incoming flow to Depot <= K
    c_depot_in = solver.Constraint(0, n_vehicles, 'depot_in')
    for i in range(1, n_nodes):
        c_depot_in.SetCoefficient(x[(i, 0)], 1)

    # 3. MTZ / Capacity Constraints
    # Standard MTZ: u_j - u_i + Capacity * x_ij <= Capacity - d_j
    # This prevents subtours and ensures capacity compliance roughly.
    for i in range(1, n_nodes):
        for j in range(1, n_nodes):
            if i != j:
                c = solver.Constraint(-infinity, float(capacity - demands[j]), f'mtz_{i}_{j}')
                c.SetCoefficient(u[j], 1)
                c.SetCoefficient(u[i], -1)
                c.SetCoefficient(x[(i, j)], capacity)

    # --- Objective ---
    objective = solver.Objective()
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j:
                objective.SetCoefficient(x[(i, j)], dist_matrix[i][j])
    objective.SetMinimization()

    # ==========================================
    # 2. HEURISTIC FUNCTION
    # ==========================================
    @lru_cache(maxsize=10000)
    def h_lp_relaxation_2_idx(state):
        unvisited = state[unvisited_var]
        
        # Optimization: Solved state
        if not unvisited: return 0.0

        # Toggle Active Nodes
        # We iterate 1..N to update bounds based on current state
        for i in range(1, n_nodes):
            if i in unvisited:
                # ACTIVE:
                # Degree must be 1 (Visited exactly once)
                cons_degree_out[i].SetBounds(1, 1)
                cons_degree_in[i].SetBounds(1, 1)
                # Load variable active
                u[i].SetBounds(demands[i], capacity)
            else:
                # INACTIVE:
                # Degree must be 0 (Removed from graph)
                cons_degree_out[i].SetBounds(0, 0)
                cons_degree_in[i].SetBounds(0, 0)
                # Load variable forced to 0
                u[i].SetBounds(0, 0)
        
        # Solve
        status = solver.Solve()
        
        if status == pywraplp.Solver.OPTIMAL:
            return float(objective.Value())
        return 0.0

    return h_lp_relaxation_2_idx

In [9]:
def dual_bound_expression_function(didp_bundle):
    """ 
    Returns a dictionary of heuristic functions (bounds) bound to the model data.
    Includes LP relaxations, Flow, MST, 1-Tree, Assignment, and Eigenvalue bounds.
    """
    
    model, metadata = didp_bundle
    
    # --- Extract metadata ---
    unvisited_var = metadata['unvisited_var']
    location_var = metadata['location_var']
    distance_list = metadata['distance_matrix']
    cost_matrix = np.array(distance_list) # Numpy version for calculations
    demand = metadata['demand']
    capacity = metadata['capacity']
    num_vehicles = metadata['num_vehicles']
    n_nodes = metadata['num_nodes'] # Ensure this exists in metadata or use len(distance_list)

    # --- Pre-computation for Bounds (Run once per problem) ---
    # masked_cost: diagonal is infinity to ignore self-loops
    masked_cost = cost_matrix.astype(float).copy()
    np.fill_diagonal(masked_cost, np.inf)
    
    # min_outgoing_arr[i] = min cost to leave node i
    min_outgoing_arr = np.min(masked_cost, axis=1)
    
    # min_incoming_arr[j] = min cost to enter node j
    min_incoming_arr = np.min(masked_cost, axis=0)

    # ==========================================
    # 1. LP Relaxation Bounds (Persistent)
    # ==========================================
    h_lp_relaxation_3_idx = create_persistent_lp_relaxation_3_index_dual_bounds(metadata=metadata)
    h_lp_relaxation_2_idx = create_persistent_lp_relaxation_2_index_dual_bounds(metadata=metadata)
    
    # ==========================================
    # 2. Flow Bound
    # ==========================================
    @lru_cache(maxsize=10000)
    def h_flow(state):
        U = state[unvisited_var]
        if not U:
            return 0.0
        
        # h_flow(U) = (2 / Q) * sum_{v in U} demand[v] * c[0][v]
        s = sum(demand[v] * distance_list[0][v] for v in U)
        h = (2.0 / capacity) * s
        return float(round(h))

    # ==========================================
    # 3. Degree Average Bound (Local Subgraph)
    # ==========================================
    @lru_cache(maxsize=10000)
    def h_degree_average(state):
        U = state[unvisited_var]
        curr = state[location_var]
        
        if not U and curr == 0:
            return 0.0

        # Active nodes: Current -> [Unvisited] -> Depot (0)
        active_nodes = [curr] + sorted(list(U))
        if 0 not in active_nodes:
            active_nodes.append(0)
            
        sub_mat = cost_matrix[np.ix_(active_nodes, active_nodes)].astype(float)
        np.fill_diagonal(sub_mat, np.inf)

        mins_in = np.min(sub_mat, axis=0) 
        mins_out = np.min(sub_mat, axis=1)
        
        # Exclude 'curr' from Incoming sum, Exclude 'depot' from Outgoing sum
        sum_in = np.sum(mins_in[1:])      
        sum_out = np.sum(mins_out[:-1])   
        
        return float(0.5 * (sum_in + sum_out))

    # ==========================================
    # 4. Global Min Flow Bound
    # ==========================================
    @lru_cache(maxsize=10000)
    def h_global_min_flow(state):
        U = state[unvisited_var]
        curr = state[location_var]
        
        if not U and curr == 0:
            return 0.0

        # Sum of minimum OUTGOING edges from (curr + U)
        val_out = sum(min_outgoing_arr[u] for u in U)
        if curr != 0:
            val_out += min_outgoing_arr[curr]
            
        # Sum of minimum INCOMING edges to (0 + U)
        val_in = sum(min_incoming_arr[u] for u in U)
        if curr != 0: 
            val_in += min_incoming_arr[0]
            
        return float(max(val_out, val_in))

    # ==========================================
    # 5. MST Bound
    # ==========================================
    @lru_cache(maxsize=10000)
    def h_mst(state):
        U = state[unvisited_var]
        if not U: return 0.0
        
        nodes = [0] + sorted(list(U))
        sub_mat = cost_matrix[np.ix_(nodes, nodes)]
        mst = minimum_spanning_tree(sub_mat)
        return float(mst.sum())

    # ==========================================
    # 6. 1-Tree Bound
    # ==========================================
    @lru_cache(maxsize=10000)
    def h_1tree(state):
        U = state[unvisited_var]
        if not U: return 0.0
        
        subset_nodes = sorted(list(U))
        depot_edges = sorted(cost_matrix[0, subset_nodes])
        e1 = depot_edges[0]
        e2 = depot_edges[1] if len(depot_edges) > 1 else 0.0
        
        if len(subset_nodes) > 1:
            sub_mat = cost_matrix[np.ix_(subset_nodes, subset_nodes)]
            mst_val = minimum_spanning_tree(sub_mat).sum()
        else:
            mst_val = 0.0 
        return float(mst_val + e1 + e2)

    # ==========================================
    # 7. Assignment Bound
    # ==========================================
    @lru_cache(maxsize=10000)
    def h_assignment(state):
        U = state[unvisited_var]
        if not U: return 0.0
        
        nodes = [0] + sorted(list(U))
        sub_mat = cost_matrix[np.ix_(nodes, nodes)]
        assign_mat = sub_mat.astype(float).copy()
        np.fill_diagonal(assign_mat, np.inf)
        
        row_ind, col_ind = linear_sum_assignment(assign_mat)
        return float(assign_mat[row_ind, col_ind].sum())

    # ==========================================
    # 8. Eigenvalue Bound
    # ==========================================
    @lru_cache(maxsize=10000)
    def h_eigen(state):
        U = state[unvisited_var]
        nodes = [0] + sorted(list(U))
        N = len(nodes)
        if N < 2: return 0.0

        D_sub = cost_matrix[np.ix_(nodes, nodes)]
        one = np.ones((N, 1))
        P = np.eye(N) - (one @ one.T) / N
        M = -P @ D_sub @ P
        M = (M + M.T) / 2
        
        try:
            eigvals = np.flip(eigh(M)[0])
        except np.linalg.LinAlgError:
            return 0.0
            
        eigvals = eigvals[np.abs(eigvals) > 1e-9]
        coeffs = np.array([1 - np.cos(2 * np.pi * k / N) for k in range(1, N)])

        phi = 0.0
        # Logic for Odd/Even N spectral bounds
        if N > 1:
            if N % 2 == 1:
                num_terms = (N - 1) // 2
                if 2 * num_terms <= len(eigvals) and num_terms <= len(coeffs):
                     phi = sum(coeffs[k-1] * (eigvals[2*k - 2] + eigvals[2*k - 1]) for k in range(1, num_terms + 1))
            else:
                num_sum_terms = N // 2 - 1
                if 2 * num_sum_terms < len(eigvals) and num_sum_terms <= len(coeffs):
                    phi = sum(coeffs[k-1] * (eigvals[2*k - 2] + eigvals[2*k - 1]) for k in range(1, num_sum_terms + 1))
                    if N-2 < len(eigvals):
                        phi += 2 * eigvals[N - 2]
                elif N > 1 and N-2 < len(eigvals):
                    phi = 2 * eigvals[N - 2]
                    
        return float(phi)

    # Return valid registry
    return automatic_creation_of_dual_bounds_registry(locals())

dual_bound_functions_registry = dual_bound_expression_function(creation_of_didp_model_function())
display(dual_bound_functions_registry)

{'h_lp_relaxation_3_idx': <functools._lru_cache_wrapper at 0x2167f656a30>,
 'h_lp_relaxation_2_idx': <functools._lru_cache_wrapper at 0x2167f6566c0>,
 'h_flow': <functools._lru_cache_wrapper at 0x2167f656ae0>,
 'h_degree_average': <functools._lru_cache_wrapper at 0x2167f656770>,
 'h_global_min_flow': <functools._lru_cache_wrapper at 0x2167f656980>,
 'h_mst': <functools._lru_cache_wrapper at 0x2167f656b90>,
 'h_1tree': <functools._lru_cache_wrapper at 0x2167f656c40>,
 'h_assignment': <functools._lru_cache_wrapper at 0x2167f656cf0>,
 'h_eigen': <functools._lru_cache_wrapper at 0x2167f656da0>}

# **Execution**

In [12]:
# ==========================================
# 1. EVOLUTIONARY ALGORITHM HYPERPARAMETERS
# ==========================================
POPULATION_SIZE = 5       # Size of the population in each generation
GENERATIONS = 1       # Number of generations to run
MUTATION_RATE = 0.2         # Probability of mutating an individual
CROSSOVER_RATE = 0.8        # Probability of performing crossover
ELITISM_RATE = 0.10
# ==========================================
# 2. OPERATOR PARAMETERS
# ==========================================
# Bounds for the coefficients generated for weighted blocks (e.g., 5.5 * h1)
LB_range_of_constant = 0.0  
UB_range_of_constant = 10.0 
# Depth limits for the RPN trees (used in Ramped Half-and-Half generator)
min_chromosome_length = 2               # Minimum depth of the initial trees
max_chromosome_length = 10               # Maximum depth of the initial trees
# Probability of selecting the best individual in the  tournament selection
# Tournament size for parent selection
tournament_size=random.randint(2, 10)
tournament_probability=0.8
# Mutation: Maximum depth allowed for the *newly generated* subtree during mutation
mutation_max_subtree_depth = random.randint(min_chromosome_length, max_chromosome_length)  # Randomly chosen between 1 and 3
# 1-Point Crossover: Probability of using Homology (matching structure) vs Random fallback
homology_1_point_crossover_probability = 0.5
# Subtree Crossover: Probability of swapping a Function (Branch) vs Terminal (Leaf)
subtree_crossover_probability = 0.9
# Uniform Crossover: Probability of swapping genes at a specific index
uniform_crossover_probability = 0.5
# ==========================================
# 4. OTHER PARAMETERS
# ==========================================
# The Ground Truth optimal cost for the specific problem instance
# Used to calculate fitness (deviation from optimal)
OPTIMAL_COST_REFERENCE= current_optimal_cost
# Time limit (in seconds) for the DIDP solver to run per chromosome evaluation
SOLVER_TIME_LIMIT = 10 #seconds

In [14]:
# B. Configure Params
params = EAHyperparameters(
    # --- 1. Population ---
    population_size=POPULATION_SIZE,          
    generations=GENERATIONS,
    crossover_rate=CROSSOVER_RATE,
    mutation_rate=MUTATION_RATE,
    elitism_rate=ELITISM_RATE,           

    # --- 2. Ranges & Constraints ---
    lb_range_of_constant=LB_range_of_constant,
    ub_range_of_constant=UB_range_of_constant,
    min_chromosome_length=min_chromosome_length,     
    max_chromosome_length=max_chromosome_length,   

    # --- 3. Operator Specifics ---
    tournament_size=tournament_size,                             
    tournament_probability=tournament_probability,                    
    mutation_max_subtree_depth=random.randint(min_chromosome_length, max_chromosome_length),                
    homology_1_point_crossover_probability=homology_1_point_crossover_probability,    
    subtree_crossover_probability=subtree_crossover_probability,             
    uniform_crossover_probability=uniform_crossover_probability,             

    # --- 4. Problem Specific ---
    reference_point=OPTIMAL_COST_REFERENCE,         
    solver_time_limit=SOLVER_TIME_LIMIT,
    
    # Optional: You can override available operations if needed
    available_operations=["ADD", "SUBTRACT", "MAX", "MIN", "MULTIPLY", "PDIV"]
)
best_ind = evolution_algorithm_execution(
    didp_model_registry=creation_of_didp_model_function,
    dual_bound_expression_function=dual_bound_expression_function,
    params=params
)

print(best_ind)

--- Initialization: Generating Population of size 5 - 1 generations ---
Generating Initial Population at time: Wed Dec 17 22:10:28 2025
-> Seeding: 1 Random Terminal + 1 Simple Subtree
Initial Population Generated at time: Wed Dec 17 22:11:18 2025
Initilization time 50.95399618148804
Initial Best Fitness: 0.18971585471969057
Gen 1: Best Fitness = 0.04977939130435211 | Global Best = 0.04977939130435211

      PERFORMANCE PROFILING REPORT      
Total Runtime:    91.8958 seconds
--- Evolution Completed ---

{'chromosome': ['h_lp_relaxation_3_idx', 'h_eigen', 'MULTIPLY', 'h_degree_average', 7.76, 'h_global_min_flow', 'MULTIPLY', 'h_assignment', 'SUBTRACT', 'MIN', 'h_lp_relaxation_2_idx', 'h_mst', 'SUBTRACT', 'ADD', 'ADD', 'h_flow', 'h_1tree', 'ADD', 'MAX'], 'fitness': 0.04977939130435211}
